# Optional Project - Colab Part3 Training

Runs Part 3: muP LR sweep on Tiny, transfer the selected LR to all muP model sizes, compare SP vs muP scaling curves, and extrapolate validation loss for a 10x larger model. Re-run interrupted training cells to resume from Google Drive checkpoints.

In [ ]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
MUP_SWEEP_CONFIG = "configs/mup_sweep_lr.yaml"
MUP_BEST_LR_JSON = "outputs/part3_mup_lr_sweep/best_lr.json"
DRIVE_MUP_BEST_LR_JSON = "/content/drive/MyDrive/svg-scaling/part3_mup_lr_sweep/best_lr.json"
SP_RUNS_DIR = "outputs/part2"
DRIVE_SP_RUNS_DIR = "/content/drive/MyDrive/svg-scaling/part2"


In [ ]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD


In [ ]:
# 2) Mount Google Drive for resumable checkpoints
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/svg-scaling/part3_mup
!mkdir -p /content/drive/MyDrive/svg-scaling/part3_mup_lr_sweep
!mkdir -p /content/drive/MyDrive/svg-scaling/part3_analysis


In [ ]:
# 3) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
# 4) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN. Public dataset loading may still work.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


In [ ]:
# 5) GPU sanity check
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('bf16_supported', torch.cuda.is_bf16_supported())


In [ ]:
# 6) Part 3 muP LR sweep on Tiny model
# If Colab disconnects, rerun this cell; each LR run resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_mup_lr_sweep.py --config {MUP_SWEEP_CONFIG}


In [ ]:
# 7) Inspect best muP LR
import json, shutil
from pathlib import Path
if not Path(MUP_BEST_LR_JSON).exists() and Path(DRIVE_MUP_BEST_LR_JSON).exists():
    Path(MUP_BEST_LR_JSON).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_MUP_BEST_LR_JSON, MUP_BEST_LR_JSON)
best = json.loads(Path(MUP_BEST_LR_JSON).read_text())
print(json.dumps(best, indent=2))
MUP_BEST_LR = best['learning_rate']
print('MUP_BEST_LR=', MUP_BEST_LR)


In [ ]:
# 8) Train all five muP model sizes for exactly one epoch with the selected Tiny LR
# If Colab disconnects, rerun this cell; each model resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_part3_mup_all.py --best-lr-json {MUP_BEST_LR_JSON}


In [ ]:
# 9) If Part 2 SP outputs are only on Drive, copy them locally for comparison
from pathlib import Path
if not list(Path(SP_RUNS_DIR).glob('*/final_metrics.json')) and Path(DRIVE_SP_RUNS_DIR).exists():
    !mkdir -p {SP_RUNS_DIR}
    !cp -r {DRIVE_SP_RUNS_DIR}/* {SP_RUNS_DIR}/
print('SP files:', len(list(Path(SP_RUNS_DIR).glob('*/final_metrics.json'))))
print('muP files:', len(list(Path('outputs/part3_mup').glob('*/final_metrics.json'))))


In [ ]:
# 10) Fit SP and muP power laws, create comparison plot/table, and extrapolate 10x larger model
%cd $REPO_DIR
!python scripts/fit_part3_scaling.py \
  --sp-runs-dir {SP_RUNS_DIR} \
  --mup-runs-dir outputs/part3_mup \
  --sp-sweep-json outputs/part2_lr_sweep/sweep_results.json \
  --mup-sweep-json outputs/part3_mup_lr_sweep/sweep_results.json \
  --output-dir outputs/part3_analysis \
  --drive-output-dir /content/drive/MyDrive/svg-scaling/part3_analysis


In [ ]:
# 11) Inspect key Part 3 outputs
from pathlib import Path
import json
for p in sorted(Path('outputs/part3_mup').glob('*/final_metrics.json')):
    m = json.loads(p.read_text())
    print(p.parent.name, {k: m.get(k) for k in ['num_parameters','val_loss','val_ppl','tokens_seen','wall_clock_seconds','peak_gpu_memory_gb']})
summary = Path('outputs/part3_analysis/part3_scaling_comparison.json')
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), indent=2))
